# Token Listing Evaluation

## Introduction

RISEx is a perpetual futures exchange that lists cryptocurrency derivatives. Every listing decision directly affects protocol solvency: an illiquid or manipulable market can amplify losses during liquidation cascades and strain the insurance vault. This notebook operationalises the RISEx token listing framework to produce a reproducible, auditable evaluation for any token declared in `config.yml`.

**Outputs:**

| Output | Description |
|---|---|
| Hard gate table | Binary pass/fail across 5 gates |
| Market score table | Six liquidity and risk metrics scored 0-100 |
| Destination tier | Prime / Major / Mid-Cap / Small-Cap / Micro-Cap |
| Soft flags | Non-blocking data quality warnings for reviewer |

**How to add a token:** append an entry to `config.yml` with fields `symbol`, `cmc_id`, `cmc_slug`, `coingecko_id`, `safety_score`, and `feed_grade`. Re-run the notebook. Data is fetched once and cached under `cache/`; set `REFRESH = True` to force a re-fetch.

**BTC and ETH** are included as reference benchmarks. Their scores represent the ceiling a token can achieve under current market conditions and calibrate what full-mark metrics look like.

---
## Methodology

Tokens are evaluated in three sequential phases. Failure at any phase is final.

### Data Sources

| Source | Endpoint | Key fields | Used for |
|---|---|---|---|
| CMC detail | `data-api/v3/cryptocurrency/detail?id={cmc_id}` | `statistics.fullyDilutedMarketCap`, `statistics.volume30d` | FDMC gate; Metric 4 (spot volume) |
| CMC historical | `data-api/v3.1/cryptocurrency/historical?id={cmc_id}&interval=24h` | `quotes[].quote.{open,high,low,close}`, `timeOpen`, `timeClose` | Metric 2 (Parkinson vol); Metric 3 (ES 95%) |
| CMC market pairs | `data-api/v3/cryptocurrency/market-pairs/latest?slug={cmc_slug}&category=spot&limit=100&sort=cmc_rank_advanced` | `marketPairs[].{exchangeSlug, quoteSymbol, volumeUsd, depthUsdNegativeTwo, depthUsdPositiveTwo, isVerified, outlierDetected, volumeExcluded}` | Binance gate; exchange score (reference); Metric 1 (spot depth, primary); Metric 5 (concentration) |
| CoinGecko tickers | `api/v3/coins/{coingecko_id}/tickers?depth=true` | `tickers[].{cost_to_move_up_usd, cost_to_move_down_usd, bid_ask_spread_percentage, market.identifier, target, is_anomaly, is_stale}` | Metric 1 (spot depth, fallback); Metric 6 (book spread) |

**OHLCV filter**: complete candles only (`timeClose < today 00:00 UTC`). Parkinson vol uses trailing 90d; ES 95% uses all available candles.

**Manual fields** (`config.yml`): `safety_score`, `feed_grade`.

---

### Phase 1: Hard Gates

| Gate | Threshold | Data source |
|---|---|---|
| FDMC | >= 100M USD | CMC detail: `statistics.fullyDilutedMarketCap` |
| Binance listing | Spot pair on Binance: USD-quoted, verified, clean | CMC market-pairs (filtered) |
| Safety score | >= 60 (60-74: conditional; >= 75: unconditional) | Manual: `config.yml safety_score` |
| Feed grade | >= Grade A | Manual: `config.yml feed_grade` |

**Binance gate rationale**: RISEx's market maker hedges exclusively on Binance. A token not listed on Binance cannot be hedged, making the listing operationally impossible regardless of market quality. Exchange score (Tier-1 = 2pts, Tier-2 = 1pt) is retained for reference and for all market quality metrics (depth, concentration) but does not gate.

**Exchange filter pipeline (applies to all market metrics)**:

1. **Fetch** - CMC market-pairs returns top 100 pairs sorted by `cmc_rank_advanced` (stable rank, not gameable 24h volume).
2. **Filter** - retain USD-quoted (USDT/USDC/USD/USDH), `isVerified=1`, `outlierDetected=0`, `volumeExcluded=0`.
3. **Aggregate** - sum `volumeUsd`, `depthUsdNegativeTwo`, `depthUsdPositiveTwo` per exchange.
4. **Tier restrict** - only Tier-1/2 exchanges are used for market quality metrics.

| Tier | Members | pts (ref) |
|---|---|---|
| Tier-1 | Binance, Coinbase (coinbase-exchange), OKX, Bybit, Kraken, Hyperliquid | 2 |
| Tier-2 | Gate, Bitget, KuCoin, Bitstamp, HTX, MEXC, BingX | 1 |
| Tier-3 | All others | 0 (excluded from metrics) |

Notes: Upbit excluded from Tier-2 — primary pairs are KRW-denominated, passes USD filter for <1/16 tested tokens. Hyperliquid in Tier-1 as on-chain CLOB; CMC does not report 2% depth for it, so depth is sourced from CoinGecko (fallback).

**Tier-1/2 restrict all market quality metrics** (depth, concentration, spread). Tier-3 venues are excluded to prevent phantom depth from obscure market-makers inflating scores.

---

### Phase 2: Market Score

$$s_i = \min\!\left(\frac{v_i}{r_i},\; 1\right) \times w_i \quad \text{(standard)} \qquad s_i = \min\!\left(\frac{r_i}{v_i},\; 1\right) \times w_i \quad \text{(linear inverted)}$$

$$s_i = \max\!\left(1 - \frac{\ln(v_i / r_i)}{\ln(v_{\max} / r_i)},\; 0\right) \times w_i \quad \text{(log inverted - ES 95% and vol only)}$$

Log scaling applies to ES 95% and Parkinson volatility only. These two metrics are approximately log-normally distributed across the crypto universe - a doubling of volatility from 50% to 100% ann. is equally as meaningful as a doubling from 100% to 200%, so equal log penalty per doubling is more appropriate than equal linear penalty per unit.

| # | Metric | Weight | Reference | v_max | Scale | Source |
|---|---|---|---|---|---|---|
| 1 | 2% Depth | 30 | 10M USD | n/a | linear | CMC market-pairs: qualified sum (h=1.0 Tier-1, h=0.5 Tier-2); CoinGecko fallback for exchanges with no CMC depth |
| 2 | Parkinson volatility | 10 | 76.4% ann. (= 4% x sqrt(365)) | 300% ann. | log | CMC historical: Parkinson estimator using daily H/L, 90d |
| 3 | ES 95% | 20 | 4% (= IMR_base Major) | 20% | log | CMC historical: close-to-close log returns, all available |
| 4 | 30d avg volume | 20 | 500M USD/day | n/a | linear | CMC detail: volume30d / 30 |
| 5 | Concentration | 10 | 0.80 (HHI <= 0.20) | n/a | linear | CMC market-pairs: 1 - HHI over Tier-1/2 |
| 6 | Book spread | 10 | 0.05% | n/a | linear | CoinGecko: median bid_ask_spread_pct over Tier-1/2 |

### Phase 3: Tier Assignment

1. **Score to unconstrained tier.** S >= 90 Prime; >= 75 Major; >= 60 Mid-Cap; >= 45 Small-Cap; >= 25 Micro-Cap; else Rejected.
2. **Feed grade ceiling.** Grade A: Prime; B: Major; C: Small-Cap; D: Micro-Cap.
3. **Prime whitelist.** Only BTC and ETH may hold Prime tier; all others demoted to Major.
4. **Listing age ceiling.** Tokens with fewer than `listing_age_min_days` days of price history (default 180) are capped at Micro-Cap regardless of score.

| Tier | Max Lev | IMR | OI Lower | OI Upper |
|---|---|---|---|---|
| Prime | 25x | 4.00% | 50M USD | 120M USD |
| Major | 20x | 5.00% | 10M USD | 20M USD |
| Mid-Cap | 10x | 10.00% | 2M USD | 10M USD |
| Small-Cap | 5x | 20.00% | 1M USD | 5M USD |
| Micro-Cap | 3x | 33.33% | 1M USD | 5M USD |

In [ ]:
import math
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

sys.path.insert(0, str(Path.cwd().parent.parent / "script"))
import token_listing_fetcher as fetcher

pd.set_option("display.max_colwidth", None)
pd.set_option("display.float_format", "{:,.4f}".format)

CONFIG_PATH = Path("config.yml")
REFRESH = True  # set True to bypass cache and re-fetch all APIs

# Load thresholds from config (falls back to defaults for any missing key)
T = fetcher.load_thresholds(CONFIG_PATH)
FDMC_MIN               = T["fdmc_min"]
EXCHANGE_SCORE_MIN     = T["exchange_score_min"]
SAFETY_PASS_MIN        = T["safety_pass_min"]
SAFETY_CONDITIONAL_MIN = T["safety_conditional_min"]
LISTING_AGE_MIN_DAYS   = T.get("listing_age_min_days", 180)

METRIC_REF = {
    1: T["metric_refs"]["spot_depth"],
    2: T["metric_refs"]["parkinson_vol"],
    3: T["metric_refs"]["es95"],
    4: T["metric_refs"]["spot_volume"],
    5: T["metric_refs"]["concentration"],
    6: T["metric_refs"]["book_spread"],
}
METRIC_WEIGHT = {
    1: T["metric_weights"]["spot_depth"],
    2: T["metric_weights"]["parkinson_vol"],
    3: T["metric_weights"]["es95"],
    4: T["metric_weights"]["spot_volume"],
    5: T["metric_weights"]["concentration"],
    6: T["metric_weights"]["book_spread"],
}
# Metrics 2 (vol) and 3 (ES 95%) use log scaling; others use linear.
METRIC_VMAX = {
    2: T["metric_vmax"]["parkinson_vol"],
    3: T["metric_vmax"]["es95"],
}
METRIC_INV = {2, 3, 6}  # inverted: lower raw value = higher score

# Build SCORE_TIERS sorted descending by threshold; append Rejected sentinel
_tier_scores = T["tier_scores"]
SCORE_TIERS = sorted(_tier_scores.items(), key=lambda kv: kv[1], reverse=True)
SCORE_TIERS = [(v, k) for k, v in SCORE_TIERS] + [(0, "Rejected")]

# Exchange tier membership — loaded from config.yml exchange_tiers section
TIER1_SLUGS, TIER2_SLUGS = fetcher.load_exchange_tiers(CONFIG_PATH)
USD_QUOTES = {"USDT", "USDC", "USD", "USDH"}

# CoinGecko market.identifier → CMC exchangeSlug (the namespace used in TIER1_SLUGS/TIER2_SLUGS).
# Only entries that differ between the two data sources are listed; identical names need no mapping.
CG_TO_CMC_SLUG: dict[str, str] = {
    "gdax":             "coinbase-exchange",
    "okex":             "okx",
    "bybit_spot":       "bybit",
    "hyperliquid-spot": "hyperliquid",
    "huobi":            "htx",
    "mxc":              "mexc",
}

PRIME_WHITELIST = {"BTC", "ETH"}
FEED_CEILING = {"A": "Prime", "B": "Major", "C": "Small-Cap", "D": "Micro-Cap"}
TIER_ORDER   = ["Prime", "Major", "Mid-Cap", "Small-Cap", "Micro-Cap", "Rejected"]
TIER_PARAMS  = {
    "Prime":     dict(max_lev=25, imr="4.00%",  mmr="2.67%",  cmr="1.78%",  oi_lower="$50M",  oi_upper="$120M"),
    "Major":     dict(max_lev=20, imr="5.00%",  mmr="3.33%",  cmr="2.22%",  oi_lower="$10M",  oi_upper="$20M"),
    "Mid-Cap":   dict(max_lev=10, imr="10.00%", mmr="6.67%",  cmr="4.44%",  oi_lower="$2M",   oi_upper="$10M"),
    "Small-Cap": dict(max_lev=5,  imr="20.00%", mmr="13.33%", cmr="8.89%",  oi_lower="$1M",   oi_upper="$5M"),
    "Micro-Cap": dict(max_lev=3,  imr="33.33%", mmr="22.22%", cmr="14.81%", oi_lower="$1M",   oi_upper="$5M"),
    "Rejected":  dict(max_lev="—", imr="—",     mmr="—",      cmr="—",      oi_lower="—",     oi_upper="—"),
}

def fmt_usd(v):
    if v >= 1e9:  return f"${v/1e9:.2f}B"
    if v >= 1e6:  return f"${v/1e6:.2f}M"
    if v >= 1e3:  return f"${v/1e3:.1f}K"
    return f"${v:.2f}"


In [ ]:
# Load config and fetch all data (cache-first)
tokens = fetcher.load_config(CONFIG_PATH)
raw = {}
for token in tokens:
    sym = token["symbol"]
    raw[sym] = fetcher.fetch_all(token, refresh=REFRESH)

def fmt_price(p):
    if p >= 1000: return f"${p:,.0f}"
    if p >= 1:    return f"${p:,.2f}"
    return f"${p:.4f}"

# --- Market overview ---
mkt_rows = []
for token in tokens:
    sym   = token["symbol"]
    stats = raw[sym]["cmc_detail"].get("data", {}).get("statistics", {})
    price = stats.get("price", 0) or 0
    mc    = stats.get("marketCap", 0) or 0
    fdv   = stats.get("fullyDilutedMarketCap", 0) or 0
    vol24 = stats.get("volume24h", 0) or stats.get("volume", 0) or 0
    vol30 = stats.get("volume30d", 0) or 0
    mkt_rows.append({
        "Token":      sym,
        "Price":      fmt_price(price),
        "Market Cap": fmt_usd(mc),
        "FDV":        fmt_usd(fdv),
        "Vol 24h":    fmt_usd(vol24),
        "Vol 30d":    fmt_usd(vol30),
        "MC/FDV":     f"{mc/fdv:.2%}" if fdv > 0 else "—",
        "Vol/MC":     f"{vol24/mc:.2%}" if mc > 0 else "—",
    })
print("Market Overview")
display(pd.DataFrame(mkt_rows).set_index("Token"))

# --- Data quality ---
# Exchange counts use the same eligibility filter as compute_exchange_gate.
qual_rows = []
for token in tokens:
    sym = token["symbol"]
    d   = raw[sym]

    t1_slugs: set[str] = set()
    t2_slugs: set[str] = set()
    other_slugs: set[str] = set()
    for pair in d["cmc_market_pairs"].get("data", {}).get("marketPairs", []):
        if pair.get("quoteSymbol") not in USD_QUOTES:                 continue
        if not pair.get("isVerified"):                                 continue
        if pair.get("outlierDetected") or pair.get("volumeExcluded"): continue
        slug = pair["exchangeSlug"]
        if slug in TIER1_SLUGS:   t1_slugs.add(slug)
        elif slug in TIER2_SLUGS: t2_slugs.add(slug)
        else:                     other_slugs.add(slug)

    quotes = d["cmc_historical"].get("data", {}).get("quotes", [])
    ohlcv_start = quotes[0].get("timeOpen", "")[:10] if quotes else "—"
    ohlcv_end   = quotes[-1].get("timeOpen", "")[:10] if quotes else "—"

    qual_rows.append({
        "Token":         sym,
        "Eligible Exch": len(t1_slugs) + len(t2_slugs) + len(other_slugs),
        "Tier-1 Exch":   len(t1_slugs),
        "Tier-2 Exch":   len(t2_slugs),
        "OHLCV Start":   ohlcv_start,
        "OHLCV End":     ohlcv_end,
        "OHLCV Days":    len(quotes),
    })
print("\nData Quality")
display(pd.DataFrame(qual_rows).set_index("Token"))

---
## Hard Gates

All gates must pass before scoring proceeds. A single failure is a hard reject.
Thresholds are loaded from `config.yml`.

In [ ]:
gate_req = pd.DataFrame([
    {"Gate": "FDMC",
     "Requirement": f">= {fmt_usd(FDMC_MIN)} fully diluted market cap"},
    {"Gate": "Binance listing",
     "Requirement": ("Spot pair on Binance: USD-quoted (USDT/USDC/USD/USDH), "
                     "isVerified=1, outlierDetected=0, volumeExcluded=0. "
                     "Required because RISEx's market maker hedges exclusively on Binance.")},
    {"Gate": "Safety score",
     "Requirement": (f">= {SAFETY_CONDITIONAL_MIN} "
                     f"({SAFETY_CONDITIONAL_MIN}-{SAFETY_PASS_MIN - 1}: conditional; "
                     f">= {SAFETY_PASS_MIN}: unconditional)")},
    {"Gate": "Feed grade",
     "Requirement": ">= Grade A (manual; Stork publisher-count integration deferred)"},
]).set_index("Gate")
display(gate_req)

print(
    "\nExchange tiering (Tier-1 = 2pts, Tier-2 = 1pt) is retained for reference "
    "and for depth/volume/concentration analysis but does not gate. "
    "CMC returns top 100 pairs sorted by cmc_rank_advanced."
)

In [ ]:
def compute_exchange_gate(market_pairs_data: dict) -> dict:
    """
    Filter, deduplicate, and score exchanges from CMC market-pairs response.

    Hard gate: token must be listed on Binance (spot, USD-quoted, verified, clean).
    Binance is required because RISEx's market maker hedges exclusively on Binance;
    no Binance listing means no hedge — automatic reject regardless of other exchanges.

    Exchange score (Tier-1 = 2pts, Tier-2 = 1pt) is retained for reference and
    for depth/volume/concentration analysis, but does not gate.

    reservesAvailable and marketReputation are not checked: all eligible exchanges
    are explicitly whitelisted in Tier-1/2, making CMC trust indicators redundant.
    """
    per_exchange: dict[str, dict] = {}
    for pair in market_pairs_data.get("data", {}).get("marketPairs", []):
        if pair.get("quoteSymbol") not in USD_QUOTES:
            continue
        if not pair.get("isVerified"):
            continue
        if pair.get("outlierDetected") or pair.get("volumeExcluded"):
            continue
        slug = pair["exchangeSlug"]
        if slug not in per_exchange:
            per_exchange[slug] = {"volume": 0.0, "depth_bid": 0.0, "depth_ask": 0.0}
        per_exchange[slug]["volume"]    += pair.get("volumeUsd", 0.0)
        per_exchange[slug]["depth_bid"] += pair.get("depthUsdNegativeTwo") or 0.0
        per_exchange[slug]["depth_ask"] += pair.get("depthUsdPositiveTwo") or 0.0

    # Tier-1/2 only subset — used for all market quality metrics.
    per_t12_exchange = {
        slug: v for slug, v in per_exchange.items()
        if slug in TIER1_SLUGS or slug in TIER2_SLUGS
    }

    # Exchange score: reference only — not used for gating.
    score = sum(
        2 if slug in TIER1_SLUGS else 1
        for slug in per_t12_exchange
    )

    # Hard gate: Binance must appear as a qualified Tier-1 pair.
    # RISEx's market maker hedges exclusively on Binance; no Binance = no hedge.
    binance_listed = "binance" in per_t12_exchange

    return {
        "score":            score,
        "binance_listed":   binance_listed,
        "pass":             binance_listed,
        "per_exchange":     per_exchange,
        "per_t12_exchange": per_t12_exchange,
    }


gates = {}
for token in tokens:
    sym       = token["symbol"]
    d         = raw[sym]
    overrides = token.get("overrides", {})
    stats     = d["cmc_detail"].get("data", {}).get("statistics", {})

    fdmc = overrides.get(
        "fullyDilutedMarketCap",
        stats.get("fullyDilutedMarketCap") or stats.get("marketCap", 0)
    )

    ex         = compute_exchange_gate(d["cmc_market_pairs"])
    safety     = token["safety_score"]
    feed_grade = token.get("feed_grade", "A")

    gates[sym] = {
        "fdmc":             fdmc,
        "fdmc_pass":        fdmc >= FDMC_MIN,
        "exchange_score":   ex["score"],
        "binance_listed":   ex["binance_listed"],
        "exchange_pass":    ex["pass"],
        "safety_score":     safety,
        "safety_pass":      safety >= SAFETY_CONDITIONAL_MIN,
        "safety_conditional": SAFETY_CONDITIONAL_MIN <= safety < SAFETY_PASS_MIN,
        "feed_grade":       feed_grade,
        "feed_pass":        feed_grade not in {"F"},
        "per_exchange":     ex["per_exchange"],
        "per_t12_exchange": ex["per_t12_exchange"],
        "all_pass":         fdmc >= FDMC_MIN and ex["pass"]
                            and safety >= SAFETY_CONDITIONAL_MIN
                            and feed_grade not in {"F"},
    }

### Hard Gate Results

In [ ]:
def gate_label(passed, conditional=False):
    if conditional: return "PASS*"
    return "PASS" if passed else "FAIL"

rows = []
for sym, g in gates.items():
    rows.append({
        "Token":        sym,
        "FDMC":         fmt_usd(g["fdmc"]),
        "FDMC gate":    gate_label(g["fdmc_pass"]),
        "Binance":      "YES" if g["binance_listed"] else "NO",
        "Binance gate": gate_label(g["exchange_pass"]),
        "Exch score":   g["exchange_score"],   # reference only
        "Safety score": g["safety_score"],
        "Safety gate":  gate_label(g["safety_pass"], g["safety_conditional"]),
        "Feed grade":   g["feed_grade"],
        "Feed gate":    gate_label(g["feed_pass"]) + " (assumed A)",
        "OVERALL":      gate_label(g["all_pass"]),
    })

gate_df = pd.DataFrame(rows).set_index("Token")
gate_cols = ["FDMC gate", "Binance gate", "Safety gate", "Feed gate", "OVERALL"]

def colour_gate(val):
    if str(val).startswith("PASS*"): return "color: orange; font-weight: bold"
    if str(val).startswith("PASS"):  return "color: green; font-weight: bold"
    if str(val).startswith("FAIL"):  return "color: red; font-weight: bold"
    return ""

display(gate_df.style.map(colour_gate, subset=gate_cols))

conditional = [s for s, g in gates.items() if g["safety_conditional"]]
if conditional:
    print(f"\nPASS* (conditional): {', '.join(conditional)} — safety score 60-74; "
          "14-day preListing mandatory; escalate to risk team.")
failed = [s for s, g in gates.items() if not g["all_pass"]]
if failed:
    print(f"Hard rejected (skipped in scoring): {', '.join(failed)}")

---
## Market Score

$$S = \sum_{i=1}^{6} s_i, \qquad S \in [0,\, 100]$$

| # | Metric | Weight | Reference (full score) | Direction |
|---|---|---|---|---|
| 1 | 2% Depth | 30 | \$10M USD | Standard |
| 2 | Parkinson Volatility | 10 | 76.4% ann. | Log inverted |
| 3 | ES 95% | 20 | 4% | Log inverted |
| 4 | 30D Avg Volume | 20 | \$500M/day | Standard |
| 5 | Concentration | 10 | 0.80 (HHI <= 0.20) | Standard |
| 6 | Book Spread | 10 | 0.05% | Linear inverted |

**Metric groupings by risk dimension:**

| Dimension | Metrics |
|---|---|
| Liquidity | Depth (M1), Volume (M4), Spread (M6) |
| Tail risk | ES 95% (M3), Parkinson Vol (M2) |
| Oracle manipulation | Volume (M4), Concentration (M5) |

Volume (M4) spans two dimensions: it is a direct liquidity measure and also sets the cost floor for oracle manipulation. Concentration (M5) complements it structurally - high volume on a single venue remains manipulable even at large scale, while volume spread across many venues does not.

---

### Metric Details

---

**1. 2% Depth** - weight 30, reference \$10M, linear

- **What:** The qualified depth: a haircut-weighted sum of (bid + ask) / 2 within +/-2% of mid-price across all Tier-1/2 venues. Tier-1 depth counts at full value (h=1.0), Tier-2 at half (h=0.5).
- **Measures:** Total liquidation absorptive capacity across qualified venues, discounting less reliable depth.
- **Source:** CMC `depthUsdNegativeTwo` / `depthUsdPositiveTwo` (primary); CoinGecko `cost_to_move_up_usd` / `cost_to_move_down_usd` as fallback for exchanges where CMC reports no depth (e.g. Hyperliquid).
- **Why chosen:**
  - Depth has the most direct causal link to cascade insolvency. When a liquidation cannot be absorbed by resting liquidity, forced selling moves the mark price, triggering further liquidations.
  - Weight 30 is the highest of any single metric, reflecting this primacy.
  - Summing across venues rewards multi-venue depth distribution; Concentration (M5) separately penalises over-reliance on a single exchange.
  - Tier-2 haircut of 50% reflects that these venues are less reliable under stress but still contribute real accessible liquidity.
  - Restricted to Tier-1/2 to exclude phantom depth from obscure venues (depth:volume ratios of 2-9x vs. 0.03-0.11x at major exchanges).

$$v_1 = \sum_{e \in \text{Tier-1/2}} h_e \cdot \frac{\textsf{bid}_{2\%,e} + \textsf{ask}_{2\%,e}}{2}, \qquad h_e = \begin{cases} 1.0 & e \in \text{Tier-1} \\ 0.5 & e \in \text{Tier-2} \end{cases}$$

Score: $s_1 = \min(v_1 / 10{,}000{,}000,\; 1) \times 30$.

---

**2. Parkinson Volatility** - weight 10, reference 76.4% ann., log inverted

- **What:** Annualised daily volatility estimated from 90-day trailing high/low ranges (Parkinson 1980 estimator).
- **Measures:** The daily margin buffer stress - the expected size of intraday price swings relative to the IMR. High vol means positions reach the liquidation trigger more frequently.
- **Why chosen:**
  - Parkinson is ~5x more efficient than close-to-close vol because intraday H/L retains all price path information.
  - 90d captures the current volatility regime without being noise-dominated by single weeks (30d) or averaged over stale cycles (365d).
  - Secondary confirmation of the daily margin buffer: captures regime risk (choppy, mean-reverting vs. trending) that ES 95% alone misses. Parkinson uses intraday H/L while ES uses only close-to-close returns, so the two metrics are largely orthogonal.
  - Reference = IMR_base(Prime) x sqrt(365) = 4% x 19.1 ~ 76.4% ann. At this vol, a 1-sigma daily move equals the tightest-tier initial margin.
  - Log scaling because vol is log-normally distributed across the crypto universe.

$$v_2 = \sqrt{\frac{365}{4n \ln 2} \sum_{t=1}^{n} \left[\ln\frac{H_t}{L_t}\right]^2}, \quad t \in \text{trailing 90 complete daily candles}$$

$\sqrt{365}$ because crypto trades 24/7. Log scoring:

$$s_2 = \max\!\left(1 - \frac{\ln(v_2 / 0.7642)}{\ln(3.0 / 0.7642)},\; 0\right) \times 10$$

$v_{\max} = 300\%$ ann. (empirical ceiling: LUNA May 2022, DOGE May 2021 both ~300-400% on 90d Parkinson). Minimum 7 days required; else $s_2 = 0$ + soft flag.

---

**3. ES 95%** - weight 20, reference 4%, log inverted

- **What:** Expected Shortfall at 95% confidence - the average of the worst 5% of daily close-to-close log returns over all available history, expressed as a positive loss fraction.
- **Measures:** Direct IMR stress test on a statistical basis: on a typical terrible day (worst 5%), how much does the token lose? If ES_95 > IMR_base, the insurance fund would be drawn on an average bad day at max leverage, not just an exceptional one.
- **Why chosen:**
  - ES_95 replaces MaxDrawdown because it uses all N x 5% tail observations (>= 20 at 400d) rather than a single peak-to-trough pair, giving far greater statistical stability.
  - MaxDrawdown is determined by exactly 2 data points and is dominated by a single extreme candle; ES_95 averages the full tail.
  - ES is a coherent risk measure (satisfies subadditivity), making it theoretically superior for portfolio-level risk aggregation.
  - Log scaling is appropriate because single-day tail losses are approximately log-normally distributed across the crypto universe.
  - Reference = IMR_base(Major) = 4%: a token whose average worst-day loss is <= 4% would not draw the insurance fund at 20x leverage on a typical bad day.

$$v_3 = -\mathbb{E}\!\left[\,r_t \;\middle|\; r_t \le \hat{q}_{0.05}\,\right], \qquad r_t = \ln\frac{C_t}{C_{t-1}}$$

where $\hat{q}_{0.05}$ is the empirical 5th percentile of daily log returns over all available complete candles. Log scoring:

$$s_3 = \max\!\left(1 - \frac{\ln(v_3 / 0.04)}{\ln(0.20 / 0.04)},\; 0\right) \times 20$$

$v_{\max} = 20\%$ (empirical ceiling for single-day ES: LUNA/DOGE 2021 type regimes). Minimum 20 returns required; else $s_3 = 0$ + soft flag. Tokens with fewer than 180 days produce an informative-only value - the listing-age Micro-Cap ceiling already handles their tier cap.

---

**4. 30D Avg Volume** - weight 20, reference \$500M/day, linear

- **What:** The 30-day average daily spot trading volume, aggregated cross-venue from CMC.
- **Measures:** Oracle manipulation cost - the capital an attacker must deploy to sustain an artificial price long enough to affect the oracle and trigger unwarranted liquidations.
- **Why chosen:**
  - Volume and Concentration (M5) jointly cover oracle manipulation resistance.
  - Volume captures the absolute scale: attacking a \$500M/day market requires deploying and absorbing hundreds of millions of dollars.
  - Weight 20 reflects that volume alone is an imperfect proxy: a \$500M/day market with 95% of volume on one venue is more manipulable than a \$200M/day market spread across five venues. Concentration (M5) corrects for this structural weakness.
  - Volume also contributes to the liquidity dimension (alongside Depth and Spread) as a measure of market activity and tradability.

$$v_4 = \frac{\textsf{volume30d}}{30}$$

Score: $s_4 = \min(v_4 / 500{,}000{,}000,\; 1) \times 20$.

---

**5. Concentration** - weight 10, reference 0.80, linear

- **What:** $1 - \textsf{HHI}$ computed over Tier-1/2 exchange volumes only, where HHI is the Herfindahl-Hirschman Index.
- **Measures:** Structural oracle manipulation resistance - how difficult it is to dominate price discovery by controlling a single venue. A score near 1 means volume is spread across many venues; near 0 means one venue dominates.
- **Why chosen:**
  - Complements Volume (M4) on the oracle manipulation dimension. A token could have high total volume but 90% concentrated on one exchange - that exchange alone can move the oracle.
  - Restricted to Tier-1/2 because those are the qualified venues that feed the price oracle; the long tail of obscure exchanges is irrelevant to manipulation risk.
  - Reference 0.80 (HHI <= 0.20) requires roughly five equally-weighted Tier-1/2 venues, at which point no single venue dominates enough to move the oracle unilaterally.

$$v_5 = 1 - \textsf{HHI}_{\text{Tier-1/2}}, \qquad \textsf{HHI} = \sum_{e \in \text{Tier-1/2}} \left(\frac{V_e}{V_{\text{total}}}\right)^2$$

Score: $s_5 = \min(v_5 / 0.80,\; 1) \times 10$.

---

**6. Book Spread** - weight 10, reference 0.05%, linear inverted

- **What:** The median bid-ask spread percentage across all Tier-1/2 exchanges, sourced from CoinGecko tickers.
- **Measures:** Upfront liquidation execution cost - the certain, unavoidable friction the exchange pays on every forced close. A 0.10% spread on a 5% IMR position consumes 2% of the margin buffer before the price moves a single basis point.
- **Why chosen:**
  - Spread is the only metric that represents a certain (not probabilistic) cost. Even in a perfectly orderly market with no adverse price movement, a wide spread still erodes the margin buffer on every liquidation.
  - Median (not mean) across exchanges reduces sensitivity to a single outlier venue.
  - Linear inversion is appropriate because spread is a direct proportional cost: 0.10% costs exactly twice as much as 0.05%.
  - Weight 10 reflects that spread is a snapshot metric (not continuous running data), so its precision is lower than depth or volume; it still carries useful signal about typical execution quality.

$$v_6 = \textsf{median}_{e \in \text{Tier-1/2}}\!\left(\overline{\textsf{bid\_ask\_spread\_pct}}_e\right) / 100$$

Score: $s_6 = \min(0.0005 / v_6,\; 1) \times 10$. Returns $0$ + soft flag if no Tier-1/2 data.


In [ ]:
from datetime import date

TODAY_UTC = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)


def _parse_quotes(historical_data: dict) -> list[dict]:
    """Extract complete daily candles (timeClose < today 00:00 UTC)."""
    quotes = historical_data.get("data", {}).get("quotes", [])
    complete = []
    for q in quotes:
        tc = q.get("timeClose", "")
        if tc:
            try:
                close_dt = datetime.fromisoformat(tc.replace("Z", "+00:00"))
                if close_dt < TODAY_UTC:
                    complete.append(q)
            except ValueError:
                pass
    return complete


def compute_parkinson_vol(historical_data: dict) -> tuple[float, int, str | None]:
    """
    Annualised Parkinson volatility, trailing 90d.

    sigma_ann = sqrt(365 / (4n * ln2) * sum(ln(H/L)^2))

    Uses daily high/low range (Parkinson 1980). ~5x more efficient than
    close-to-close because the intraday range retains path information
    that close-to-close discards.

    Returns (sigma_ann, n_days_used, flag_or_None).
    """
    quotes = _parse_quotes(historical_data)[-90:]
    n = len(quotes)
    if n < 7:
        return 0.0, n, f"insufficient history ({n}d < 7d); vol scored 0"
    hl_sq = []
    for q in quotes:
        h = q["quote"]["high"]
        l = q["quote"]["low"]
        if h > 0 and l > 0 and h >= l:
            hl_sq.append(math.log(h / l) ** 2)
    n_valid = len(hl_sq)
    if n_valid < 7:
        return 0.0, n_valid, f"insufficient valid candles ({n_valid}d < 7d); vol scored 0"
    sigma_ann = math.sqrt(365 * sum(hl_sq) / (4 * n_valid * math.log(2)))
    flag = f"short history ({n}d < 90d)" if n < 90 else None
    return sigma_ann, n, flag


def compute_es95(historical_data: dict) -> tuple[float, int, str | None]:
    """
    Expected Shortfall at 95% confidence over all available daily candles.

    ES_95 = mean of the worst 5% of close-to-close daily log returns,
    expressed as a positive loss fraction.

    Uses all available history (not capped) to maximise tail observations:
    at 400d, 5% tail = 20 observations — the statistical minimum for a
    stable ES estimate.

    Tokens below 180d still produce a value; the flag marks it informative
    only — the listing-age Micro-Cap ceiling already handles their tier cap.

    Returns (es95, n_returns, flag_or_None).
    """
    quotes = _parse_quotes(historical_data)
    closes = [q["quote"]["close"] for q in quotes if q["quote"].get("close", 0) > 0]
    n_closes = len(closes)
    if n_closes < 21:  # need >= 20 returns (21 closes)
        return 0.0, max(n_closes - 1, 0), f"insufficient history ({n_closes - 1}d < 20d); ES scored 0"
    log_returns = np.diff(np.log(closes))
    cutoff = np.percentile(log_returns, 5)
    tail = log_returns[log_returns <= cutoff]
    es95 = float(-np.mean(tail))  # positive loss fraction
    n_returns = len(log_returns)
    flag = f"ES_INFORMATIVE: {n_returns}d < 180d; value informative only" if n_returns < 180 else None
    return es95, n_returns, flag


def build_cg_depth(cg_tickers: dict) -> dict[str, dict]:
    """
    Build per-exchange 2% depth from CoinGecko tickers (fetched with depth=true).

    CoinGecko returns cost_to_move_up_usd (ask-side 2% depth) and
    cost_to_move_down_usd (bid-side 2% depth) per ticker. Values are summed
    across all USD-quoted, non-anomalous, non-stale pairs per exchange.

    CG market identifiers are translated to CMC exchangeSlug namespace via
    CG_TO_CMC_SLUG so the returned dict is compatible with TIER1_SLUGS/TIER2_SLUGS.
    """
    per_exchange: dict[str, dict] = {}
    for t in cg_tickers.get("tickers", []):
        if t.get("target") not in USD_QUOTES:
            continue
        if t.get("is_anomaly") or t.get("is_stale"):
            continue
        cg_ident = t.get("market", {}).get("identifier", "")
        slug = CG_TO_CMC_SLUG.get(cg_ident, cg_ident)
        bid = t.get("cost_to_move_down_usd") or 0.0
        ask = t.get("cost_to_move_up_usd") or 0.0
        if bid == 0.0 and ask == 0.0:
            continue
        if slug not in per_exchange:
            per_exchange[slug] = {"depth_bid": 0.0, "depth_ask": 0.0}
        per_exchange[slug]["depth_bid"] += bid
        per_exchange[slug]["depth_ask"] += ask
    return per_exchange


def compute_spot_depth(per_t12_exchange: dict, cg_depth: dict | None = None,
                        hl_depth: dict | None = None) -> float:
    """
    Qualified depth: haircut-weighted sum of (bid + ask) / 2 across Tier-1/2 exchanges.

    CMC depth (depthUsdNegativeTwo/PositiveTwo) is used when non-zero;
    Hyperliquid L2 API depth is next (authoritative for the "hyperliquid" slug);
    CoinGecko depth is the final fallback.

    h_e = 1.0 for Tier-1, 0.5 for Tier-2.
    QualifiedDepth = sum(h_e * (bid_e + ask_e) / 2),  priority: CMC > HL > CG

    Tier-2 depth counts at 50% to reflect lower reliability under stress.
    Summing across venues rewards multi-venue distribution; adding any qualified
    venue never decreases the score. The concentration metric (M5) separately
    penalises over-reliance on a single exchange.
    """
    total = 0.0
    for slug, v in per_t12_exchange.items():
        if v["depth_bid"] > 0 or v["depth_ask"] > 0:
            bid, ask = v["depth_bid"], v["depth_ask"]
        else:
            hl = hl_depth.get(slug) if hl_depth else None
            cg = cg_depth.get(slug) if cg_depth else None
            bid = (hl or cg or {}).get("depth_bid", 0.0)
            ask = (hl or cg or {}).get("depth_ask", 0.0)
        d = (bid + ask) / 2
        h = 1.0 if slug in TIER1_SLUGS else 0.5
        total += h * d
    return total


def compute_hhi_concentration(per_t12_exchange: dict) -> float:
    """
    1 - HHI over Tier-1/2 exchange volumes.

    HHI over all exchanges is uninformative (every token scores ~0.85-0.93).
    Restricting to Tier-1/2 captures oracle manipulation risk: an attacker must
    move the price on the qualified venues that feed the oracle.
    """
    total = sum(v["volume"] for v in per_t12_exchange.values())
    if total == 0:
        return 0.0
    hhi = sum((v["volume"] / total) ** 2 for v in per_t12_exchange.values())
    return 1.0 - hhi


def compute_book_spread(cg_tickers: dict, t12_slugs: set[str]) -> tuple[float, str | None]:
    """
    Median bid_ask_spread_percentage across all Tier-1/2 exchanges (CoinGecko).

    Median is used instead of mean to reduce sensitivity to a single outlier exchange.
    Restricted to Tier-1/2 for consistency with depth and concentration metrics.
    CG identifiers are translated to CMC slugs via CG_TO_CMC_SLUG before matching.
    Returns (median_spread_fraction, flag_or_None).
    """
    tickers = cg_tickers.get("tickers", [])
    if not tickers:
        return 0.0, "CG_NO_DATA: no tickers returned; spread scored 0"

    per_exchange: dict[str, list[float]] = {}
    for t in tickers:
        cg_ident = t.get("market", {}).get("identifier", "")
        slug = CG_TO_CMC_SLUG.get(cg_ident, cg_ident)
        if slug not in t12_slugs:
            continue
        if t.get("target") not in USD_QUOTES:
            continue
        if t.get("is_anomaly") or t.get("is_stale"):
            continue
        spread_pct = t.get("bid_ask_spread_percentage")
        if spread_pct is None or spread_pct <= 0 or spread_pct > 500:
            continue
        per_exchange.setdefault(slug, []).append(spread_pct)

    if not per_exchange:
        return 0.0, "CG_NO_MATCH: no Tier-1/2 exchanges matched in CoinGecko tickers; spread scored 0"

    exchange_medians = [float(np.median(spreads)) for spreads in per_exchange.values()]
    median_spread_pct = float(np.median(exchange_medians))
    # Convert percentage to fraction (e.g. 0.0105 -> 0.000105)
    return median_spread_pct / 100.0, None


def score_metric(v: float, ref: float, w: int, inverted: bool,
                 v_max: float | None = None) -> float:
    """
    Score a single metric.

    Linear (v_max=None):
      standard:  min(v/ref, 1) * w
      inverted:  min(ref/v, 1) * w

    Log-scaled (v_max provided, inverted only -- ES 95% and vol):
      s = max(1 - ln(v/ref) / ln(v_max/ref), 0) * w
      Full score at v <= ref; zero at v >= v_max; log-linear decay between.
      Log scaling is appropriate here because these metrics are approximately
      log-normally distributed across the crypto asset universe, and their
      references are anchored to a financial threshold (IMR_base), so equal
      penalty per doubling is more meaningful than equal penalty per unit.
    """
    if v <= 0:
        return 0.0
    if not inverted:
        return min(v / ref, 1.0) * w
    # inverted
    if v <= ref:
        return float(w)
    if v_max is not None:
        # log scaling
        if v >= v_max:
            return 0.0
        return max(1.0 - math.log(v / ref) / math.log(v_max / ref), 0.0) * w
    # linear inverted
    return min(ref / v, 1.0) * w


In [ ]:
scores: dict[str, dict] = {}
flags:  dict[str, list] = {}  # sym -> [(metric_id, description), ...]

for token in tokens:
    sym = token["symbol"]
    if not gates[sym]["all_pass"]:
        continue

    d               = raw[sym]
    overrides       = token.get("overrides", {})
    per_t12_exchange = gates[sym]["per_t12_exchange"]
    t12_slugs       = set(per_t12_exchange.keys())
    token_flags: list[tuple[str, str]] = []

    # Metric 1: Spot depth — CMC primary, HL L2 API secondary, CG fallback
    cg_depth = build_cg_depth(d["coingecko_tickers"])
    hl_depth = {"hyperliquid": d["hl_depth"]} if d.get("hl_depth") else None
    m1_val = compute_spot_depth(per_t12_exchange, cg_depth, hl_depth)
    s1 = score_metric(m1_val, METRIC_REF[1], METRIC_WEIGHT[1], False)

    # Metric 2: Parkinson volatility — log-scaled, ref = IMR_base(Major) x sqrt(365)
    sigma, n_v, flag2 = compute_parkinson_vol(d["cmc_historical"])
    if flag2: token_flags.append(("m2_vol", flag2))
    s2 = score_metric(sigma, METRIC_REF[2], METRIC_WEIGHT[2], True, METRIC_VMAX[2])

    # Metric 3: ES 95% — all available daily candles, log-scaled, ref = IMR_base(Major) = 5%
    es95, n_d, flag3 = compute_es95(d["cmc_historical"])
    if flag3: token_flags.append(("m3_es95", flag3))
    s3 = score_metric(es95, METRIC_REF[3], METRIC_WEIGHT[3], True, METRIC_VMAX[3])

    # Metric 4: Spot volume — cross-venue 30d avg daily
    stats = d["cmc_detail"].get("data", {}).get("statistics", {})
    volume30d = overrides.get("volume30d", stats.get("volume30d", 0) or 0)
    avg_daily = volume30d / 30
    s4 = score_metric(avg_daily, METRIC_REF[4], METRIC_WEIGHT[4], False)

    # Metric 5: Concentration — 1-HHI over Tier-1/2 only
    conc = compute_hhi_concentration(per_t12_exchange)
    s5 = score_metric(conc, METRIC_REF[5], METRIC_WEIGHT[5], False)

    # Metric 6: Book spread — median across Tier-1/2, CoinGecko
    spread, flag6 = compute_book_spread(d["coingecko_tickers"], t12_slugs)
    if flag6: token_flags.append(("m6_spread", flag6))
    s6 = score_metric(spread, METRIC_REF[6], METRIC_WEIGHT[6], True)

    total = s1 + s2 + s3 + s4 + s5 + s6
    scores[sym] = {
        "s1": s1, "v1": m1_val,
        "s2": s2, "v2": sigma,  "n_v": n_v,
        "s3": s3, "v3": es95,   "n_d": n_d,
        "s4": s4, "v4": avg_daily,
        "s5": s5, "v5_hhi": 1 - conc,
        "s6": s6, "v6": spread,
        "total": total,
    }
    flags[sym] = token_flags


### Metric Scores

In [ ]:
w = METRIC_WEIGHT
rows = []
for sym, sc in scores.items():
    rows.append({
        "Token":                       sym,
        f"2% Depth ({w[1]})":          f"{sc['s1']:.1f}  [{fmt_usd(sc['v1'])}]",
        f"Parkinson Vol ({w[2]})":     f"{sc['s2']:.1f}  [{sc['v2']*100:.0f}% ann, {sc['n_v']}d]",
        f"ES 95% ({w[3]})":            f"{sc['s3']:.1f}  [{sc['v3']*100:.2f}%, {sc['n_d']}d]",
        f"30D Avg Volume ({w[4]})":    f"{sc['s4']:.1f}  [{fmt_usd(sc['v4'])}/d]",
        f"Concentration ({w[5]})":     f"{sc['s5']:.1f}  [HHI={sc['v5_hhi']:.3f}]",
        f"Book Spread ({w[6]})":       f"{sc['s6']:.1f}  [{sc['v6']*100:.4f}%]",
        "Total S":                     f"{sc['total']:.1f}",
    })

display(pd.DataFrame(rows).set_index("Token"))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

syms = list(scores.keys())
n    = len(syms)
x    = np.arange(n)

v1 = [scores[s]["v1"] / 1e6           for s in syms]
v2 = [scores[s]["v2"] * 100           for s in syms]
v3 = [scores[s]["v3"] * 100           for s in syms]
v4 = [scores[s]["v4"] / 1e6           for s in syms]
v5 = [(1 - scores[s]["v5_hhi"])       for s in syms]
v6 = [scores[s]["v6"] * 100           for s in syms]

_REF = [
    (METRIC_REF[1] / 1e6,  False, f"M1: 2% Spot Depth (w={METRIC_WEIGHT[1]})",        "$M"),
    (METRIC_REF[2] * 100,  True,  f"M2: Parkinson Vol 90d (w={METRIC_WEIGHT[2]})",    "% ann."),
    (METRIC_REF[3] * 100,  True,  f"M3: ES 95% (w={METRIC_WEIGHT[3]})",               "%"),
    (METRIC_REF[4] / 1e6,  False, f"M4: Avg Daily Vol 30d (w={METRIC_WEIGHT[4]})",    "$M"),
    (METRIC_REF[5],        False, f"M5: Concentration T1/2 (w={METRIC_WEIGHT[5]})",   ""),
    (METRIC_REF[6] * 100,  True,  f"M6: Book Spread T1/2 (w={METRIC_WEIGHT[6]})",    "%"),
]
_VALS = [v1, v2, v3, v4, v5, v6]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle("Raw Metric Values vs Reference Thresholds", fontsize=13, fontweight="bold")

for ax, vals, (ref_val, inverted, title, unit) in zip(axes.flat, _VALS, _REF):
    colors = [
        "#2ecc71" if (v <= ref_val if inverted else v >= ref_val) else "#e74c3c"
        for v in vals
    ]
    ax.bar(x, vals, color=colors, alpha=0.85, width=0.6)
    ax.axhline(ref_val, color="navy", linestyle="--", linewidth=1.2, label=f"Ref: {ref_val:.3g}")
    ax.set_xticks(x)
    ax.set_xticklabels(syms, rotation=45, ha="right", fontsize=9)
    ax.set_title(title, fontsize=10)
    if unit:
        ax.set_ylabel(unit, fontsize=9)
    ax.legend(fontsize=8)
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

# Stacked score breakdown
_COMPONENTS  = ["s1", "s2", "s3", "s4", "s5", "s6"]
_COMP_LABELS = [
    f"Depth ({METRIC_WEIGHT[1]})",
    f"Parkinson Vol ({METRIC_WEIGHT[2]})",
    f"ES 95% ({METRIC_WEIGHT[3]})",
    f"Volume ({METRIC_WEIGHT[4]})",
    f"Conc ({METRIC_WEIGHT[5]})",
    f"Spread ({METRIC_WEIGHT[6]})",
]
_PALETTE = ["#3498db", "#9b59b6", "#e67e22", "#2ecc71", "#1abc9c", "#e74c3c"]

fig2, ax2 = plt.subplots(figsize=(12, 5))
bottom = np.zeros(n)
for key, label, color in zip(_COMPONENTS, _COMP_LABELS, _PALETTE):
    comp_vals = np.array([scores[s][key] for s in syms])
    ax2.bar(x, comp_vals, bottom=bottom, label=label, color=color, alpha=0.88, width=0.6)
    bottom += comp_vals

for xi, tot in zip(x, bottom):
    ax2.text(xi, tot + 0.8, f"{tot:.0f}", ha="center", va="bottom", fontsize=9, fontweight="bold")

tier_lines = [
    (_tier_scores["Prime"],     "gold",   "--", f"Prime ({_tier_scores['Prime']})"),
    (_tier_scores["Major"],     "silver", "--", f"Major ({_tier_scores['Major']})"),
    (_tier_scores["Mid-Cap"],   "gray",   ":",  f"Mid-Cap ({_tier_scores['Mid-Cap']})"),
    (_tier_scores["Small-Cap"], "brown",  ":",  f"Small-Cap ({_tier_scores['Small-Cap']})"),
]
for threshold, color, ls, label in tier_lines:
    ax2.axhline(threshold, color=color, linestyle=ls, linewidth=1.2, label=label)

ax2.set_xticks(x)
ax2.set_xticklabels(syms, rotation=0, fontsize=10)
ax2.set_ylabel("Score contribution")
ax2.set_ylim(0, 112)
ax2.set_title("Market Score Breakdown by Metric", fontsize=13, fontweight="bold")
ax2.legend(loc="upper right", fontsize=9, ncol=4)
ax2.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---
## Tier Assignment

Four steps applied in order:

1. **Score to unconstrained tier**: S >= 90 → Prime, >= 75 → Major, >= 60 → Mid-Cap, >= 45 → Small-Cap, >= 25 → Micro-Cap, else → Rejected.
2. **Apply feed grade ceiling**: destination = min(score tier, grade ceiling).
3. **Prime whitelist**: only BTC and ETH are Prime-eligible; all others are demoted to Major if Step 2 yields Prime.
4. **Listing age ceiling**: tokens with fewer than `listing_age_min_days` days of price history (default 180) are capped at Micro-Cap regardless of score.

**Threshold rationale:**

The 15-point gaps between tiers correspond roughly to losing one major metric's full contribution, or material underperformance across 2-3 smaller metrics. With weights summing to 100, a 15-point drop means one full metric (e.g., spread at 10pts, or vol at 10pts, or half of ES at 10pts) is lost plus some degradation elsewhere.

- **Prime (90):** Near-full scores across all metrics. The 10-point margin from 100 tolerates only minor shortfalls (spread slightly above reference, concentration not at maximum). In practice, only BTC and ETH reliably reach this tier - the Prime whitelist enforces this explicitly.
- **Major (75):** Strong depth (M1 = 30) is almost required; without it the maximum achievable is 70. The remaining 45 points must come from solid ES/Vol and Volume performance. One weak metric is tolerable; two weak metrics typically drop to Mid-Cap.
- **Mid-Cap (60):** Meaningful market presence with notable deficiencies. Depth may score around 15 (half-reference); the rest of the metrics must compensate. Suitable for 10x max leverage.
- **Small-Cap (45):** Multiple metrics underperform reference. Only marginal exchange presence. Suitable for 5x max leverage and small OI caps only.
- **Micro-Cap (25):** Minimum passing bar. Most metrics are weak; token has some exchange presence but fails most references. Listed at 3x max leverage only.
- **Rejected (<25):** No credible listing case. Insufficient liquidity or risk profile to safely list even at minimum leverage.

In [ ]:
def score_to_tier(s: float) -> str:
    for threshold, tier in SCORE_TIERS:
        if s >= threshold:
            return tier
    return "Rejected"


def apply_feed_ceiling(tier: str, ceiling: str) -> str:
    """Return the more conservative (lower) of tier and ceiling."""
    i_tier    = TIER_ORDER.index(tier)
    i_ceiling = TIER_ORDER.index(ceiling)
    return tier if i_tier >= i_ceiling else ceiling


def recommend_prelisting(tier: str, feed_grade: str, score: float,
                         safety_conditional: bool) -> str:
    if safety_conditional:
        return "14d (safety conditional)"
    if tier in {"Prime", "Major"} and feed_grade == "A" and score >= 80:
        return "24h"
    if tier in {"Mid-Cap", "Small-Cap"} or feed_grade == "B":
        return "7d"
    return "14d"


tiers: dict[str, dict] = {}
for sym, sc in scores.items():
    raw_tier = score_to_tier(sc["total"])
    ceiling  = FEED_CEILING.get(gates[sym]["feed_grade"], "Micro-Cap")
    capped   = apply_feed_ceiling(raw_tier, ceiling)
    final    = "Major" if capped == "Prime" and sym not in PRIME_WHITELIST else capped

    # Listing age ceiling: tokens with fewer than LISTING_AGE_MIN_DAYS of OHLCV data
    # are capped at Micro-Cap regardless of score. Configurable in config.yml.
    token_age_days = len(_parse_quotes(raw[sym]["cmc_historical"]))
    age_capped = token_age_days < LISTING_AGE_MIN_DAYS
    if age_capped:
        final = apply_feed_ceiling(final, "Micro-Cap")

    tiers[sym] = {
        "raw_tier":          raw_tier,
        "feed_ceiling":      ceiling,
        "capped_tier":       capped,
        "final_tier":        final,
        "whitelist_demoted": capped == "Prime" and sym not in PRIME_WHITELIST,
        "age_capped":        age_capped,
        "token_age_days":    token_age_days,
    }

---
## Summary

In [ ]:
summary_rows = []
for token in tokens:
    sym = token["symbol"]
    g   = gates[sym]

    if not g["all_pass"]:
        failed_gates = []
        if not g["fdmc_pass"]:       failed_gates.append("FDMC")
        if not g["exchange_pass"]:   failed_gates.append("Exchange")
        if not g["safety_pass"]:     failed_gates.append("Safety")
        if not g["feed_pass"]:       failed_gates.append("Feed")
        summary_rows.append({
            "Token": sym, "Gate": "FAIL", "Score S": "—",
            "Raw Tier": "—", "Feed Grade": g["feed_grade"],
            "Destination Tier": f"REJECTED ({', '.join(failed_gates)})",
            "Max Lev": "—", "IMR": "—", "OI Lower": "—", "OI Upper": "—",
        })
        continue

    sc   = scores[sym]
    tier = tiers[sym]
    p    = TIER_PARAMS[tier["final_tier"]]
    dest_label = tier["final_tier"]
    if tier["whitelist_demoted"]:
        dest_label += " [whitelist cap]"
    if tier["age_capped"]:
        dest_label += f" [age cap: {tier['token_age_days']}d < {LISTING_AGE_MIN_DAYS}d]"

    summary_rows.append({
        "Token":            sym,
        "Gate":             "PASS*" if g["safety_conditional"] else "PASS",
        "Score S":          f"{sc['total']:.1f}",
        "Raw Tier":         tier["raw_tier"],
        "Feed Grade":       g["feed_grade"],
        "Destination Tier": dest_label,
        "Max Lev":          f"{p['max_lev']}x",
        "IMR":              p["imr"],
        "OI Lower":         p["oi_lower"],
        "OI Upper":         p["oi_upper"],
    })

summary_df = pd.DataFrame(summary_rows).set_index("Token")

def colour_summary(val):
    if str(val).startswith("PASS*"): return "color: orange; font-weight: bold"
    if str(val).startswith("PASS"):  return "color: green; font-weight: bold"
    if str(val).startswith("FAIL") or str(val).startswith("REJECT"): return "color: red; font-weight: bold"
    return ""

display(summary_df.style.map(colour_summary, subset=["Gate", "Destination Tier"]))


---
## Soft Flags

Non-blocking. Attached to the listing record for reviewer reference.

| Flag | Meaning |
|---|---|
| `m2_vol: short history` | Volatility computed on < 90d; estimate is noisier |
| `m2_vol: insufficient history` | < 7d data; vol scored 0 |
| `m3_es95: ES_INFORMATIVE` | ES 95% computed on < 180d; value informative only - Micro-Cap ceiling already applies |
| `m3_es95: insufficient history` | < 20 returns; ES scored 0 |
| `m6_spread: CG_NO_DATA` | CoinGecko returned no tickers; spread scored 0 (10 pts missing) |
| `m6_spread: CG_NO_MATCH` | No Tier-1/2 exchanges matched in CoinGecko response; spread scored 0 |
| `age_cap` | Token has fewer than `listing_age_min_days` days of price history; tier capped at Micro-Cap |

In [ ]:
any_flags = False
for sym, sym_flags in flags.items():
    if sym_flags:
        any_flags = True
        print(f"{sym}:")
        for ft, fd in sym_flags:
            print(f"  [{ft}] {fd}")
if not any_flags:
    print("No soft flags.")